# Notebook 37 — Accuracy vs. profitability: target-alignment research

**Research-only; no live orders.** Compare 3-class direction classification, next-bar return regression, and cost-aware binary classification on identical expanding chronological validation folds. All models use features available at the bar timestamp. The closed final 20% historical holdout and prospective ledger are excluded.

**Important execution assumptions:** `next_return` is a close-to-close adjusted return, not necessarily an executable next-open-to-next-close return. Cost assumptions are illustrative, and overlapping daily signals are not an executable portfolio backtest. We report equal-weight daily returns with unfilled slots in cash and no compounding across folds. Target construction and model fitting use training labels only.

In [ ]:
from pathlib import Path
import sys, time
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
assert (ROOT/'src').is_dir(), 'Open this notebook from the repository or notebooks directory'
sys.path.insert(0,str(ROOT/'src'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.classifier_benchmark import eligible_universe
from stock_ml_dynamic.feature_ablation import development_holdout_split
from stock_ml_dynamic.walk_forward import expanding_folds, fold_data
from stock_ml_dynamic.historical_market import add_historical_market_features
from stock_ml_dynamic.accuracy_features import add_accuracy_features
from stock_ml_dynamic.volume_regime_research import add_volume_regime_features
from stock_ml_dynamic.economic_target_research import run_target_comparison, summarize_research
SOURCE=ROOT/'artifacts/master_tables/1d'
required=['master_labeled.parquet','latest_inference.parquet','transformers.joblib','manifest.json']
missing=[x for x in required if not (SOURCE/x).exists()]
if missing: raise FileNotFoundError(f'Missing {missing} under {SOURCE}. Check notebook 03 output path.')
master,latest,bundle,manifest=load_master(SOURCE)
print('Master:',master.shape,'Tickers:',master.symbol.nunique(),'Dates:',master.bar_timestamp.min(),master.bar_timestamp.max())

## Experiment configuration

- **Outer folds:** 3 expanding chronological development folds, target labels re-fit within each fold (`q=7`).
- **Models:** Random Forest 3-class direction, HistGradientBoostingRegressor next return, HistGradientBoostingClassifier net-positive event.
- **Economic target:** next return strictly greater than the assumed round-trip transaction cost (10 bps); this is a binary classification target, **not** the Q7 Up label.
- **Decision policies:** direction: predict Up with P(Up) ≥ 0.40; regression: predicted return > 10 bps; economic binary: P(net positive) ≥ 0.50. All use at most 10 positions per day, ranking by model score; empty slots remain cash. These thresholds are predeclared, not optimized on validation data.
- **Metrics:** 3-class accuracy/ROC-AUC, return MAE/RMSE/R², cost-aware event ROC-AUC/Brier, entry precision, trade mean net return, daily portfolio net return, utilization, drawdown, and paired daily differences.
- **Caveat:** differing thresholds produce different trade counts. This is a policy comparison, not an exact-coverage ranking comparison.

In [ ]:
SEED=42
COST_BPS=10
MAX_POSITIONS=10
CAP=60_000
RF_TREES=160
OUTPUT=ROOT/'artifacts/target_alignment/1d'
print('Configuration:',dict(cost_bps=COST_BPS,max_positions=MAX_POSITIONS,train_cap=CAP,rf_trees=RF_TREES))

## Load the same development universe and causal features as earlier research

In [ ]:
eligible,eligibility=eligible_universe(master)
development,cutoff=development_holdout_split(eligible,.20)
expanded,hist=add_historical_market_features(development)
expanded,technical=add_accuracy_features(expanded)
expanded,volume=add_volume_regime_features(expanded)
feature_cols=list(dict.fromkeys(hist+technical+volume))
print('Development rows:',len(development),'boundary:',cutoff,'additional causal features:',len(feature_cols))
assert pd.to_datetime(expanded.bar_timestamp,utc=True).max()<=cutoff

## Three paired outer-fold experiments

The same validation rows are scored by all three models. For each fold, the last training outcome timestamp is at or before the training boundary, and validation features start after it. Missing and extreme engineered values are imputed/clipped using **training-only** statistics.

In [ ]:
start=time.perf_counter()
result=run_target_comparison(expanded,feature_cols,n_folds=3,cap=CAP,rf_trees=RF_TREES,
                             cost_bps=COST_BPS,max_positions=MAX_POSITIONS,seed=SEED,
                             output_dir=OUTPUT)
print('Elapsed seconds:',round(time.perf_counter()-start,1))
print('Files saved to:',OUTPUT)
print('Leakage audit:')
display(result['audit'])

## Predictive metrics: compare each model against its own target

Do not compare 3-class accuracy directly with binary accuracy: their base rates and loss functions differ. Focus on common **economic** outcomes in the next section.

In [ ]:
display(result['predictive'].round(6))
display(result['predictive'].groupby('model').mean(numeric_only=True).round(6))

## Same-day portfolio policy results

Returns are reported as decimal fractions (e.g., 0.001 = 0.1%). Equal capital across ten slots; idle slots are cash. One-day signal horizon; this simplified research simulation does not model execution slippage, fills, borrow, financing, market impact, or intraday availability.

In [ ]:
display(result['portfolio'].round(6))
display(result['portfolio'].groupby('model').mean(numeric_only=True).round(6))
fig,axes=plt.subplots(1,2,figsize=(12,4))
result['portfolio'].pivot(index='fold',columns='model',values='mean_daily_net').plot(kind='bar',ax=axes[0],title='Daily portfolio net return by fold')
result['portfolio'].pivot(index='fold',columns='model',values='utilization').plot(kind='bar',ax=axes[1],title='Capital utilization by fold')
plt.tight_layout();plt.show()

## Return magnitude versus label correctness

A direction classifier can be correct about the Q7 Up class but still lose money after costs. Conversely, a binary net-positive classifier directly targets the sign of *realized net return*, while regression targets magnitude. The following diagnostics quantify the difference.

In [ ]:
display(result['selection'].round(6))
display(result['paired_daily'].round(7))
fig,ax=plt.subplots(figsize=(10,4))
for name,g in result['daily'].groupby('model'):
    avg=g.groupby('bar_timestamp')['portfolio_net'].mean().sort_index()
    ax.plot(avg.index,avg.cumsum(),label=name)
ax.set_title('Cumulative sum of daily net returns (diagnostic, not compounded)')
ax.set_ylabel('Cumulative decimal return');ax.legend();plt.tight_layout();plt.show()

## Interpretation checklist

1. Does the return regressor achieve lower MAE than a training-mean baseline and positive out-of-sample R²? A model can be useful for ranking even with negative R², but that requires independent economic evidence.
2. Does the economic classifier produce positive mean net returns **after costs** and remain active in every fold?
3. Are paired daily return improvements consistent across all folds, rather than concentrated in one market regime?
4. Is the strategy investable given execution timing, slippage, concentration, and turnover? This notebook does **not** answer that last question.

**Do not use the final historical holdout for additional tuning.** The prospective ledger is not read or written.

In [ ]:
summary=summarize_research(result)
print(summary)
print('Saved artifacts:',*[p.name for p in sorted(OUTPUT.glob('*.csv'))],sep='\n  ')